In [ ]:
import pandas as pd

df = pd.read_csv('./StormEvents_details-ftp_v1.0_d1950_c20250520.csv')

df.columns

In [ ]:
from process_noaa_data import map_to_ef_scale

df = map_to_ef_scale(df)
df.columns

# Required Features
cape,cin,dewpoint_2m,temp_2m,tcwv,surface_pressure,shear_0_1km,shear_0_3km,cin_missing


In [ ]:
import cdsapi

c = cdsapi.Client()

dataset = "reanalysis-era5-single-levels"
filename = "era5_single_1950_04.nc"

request = {
    "product_type": "reanalysis",
    "format": "netcdf",  # best practice for Python/xarray
    "variable": [
        "2m_temperature",
        "2m_dewpoint_temperature",
        "surface_pressure",
        "total_column_water_vapour",
        "convective_available_potential_energy",
        "convective_inhibition",
    ],
    "year": "1950",
    "month": "04",
    "day": ["28", "29"],
    # Best practice: request the full day(s) once, then sample locally
    "time": [f"{h:02d}:00" for h in range(24)],
    # Best practice: restrict area to your tornado bounding box (+margin)
    # CDS uses [North, West, South, East]
    "area": [36.5, -100.5, 30.5, -97.5],  # example bbox; set from your events
}

c.retrieve(dataset, request, filename)

2025-12-21 15:12:47,534 INFO [2025-12-03T00:00:00Z] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
2025-12-21 15:12:48,286 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2025-12-21 15:12:48,287 INFO Request ID is b8c7a2ff-80d5-438a-94a5-d95c18f80f96
2025-12-21 15:12:48,491 INFO status has been updated to accepted
2025-12-21 15:13:02,738 INFO status has been updated to running
2025-12-21 15:14:05,340 INFO status has been updated to successful


'era5_single_1950_04.nc'

In [11]:
from process_noaa_data import map_to_ef_scale, open_nc
ds = open_nc("era5_single_1950_04.nc")
df = ds.to_dataframe()
df.head()

number expver         t2m         d2m  \
valid_time latitude longitude                                          
1950-04-28 36.5     -100.50         0   0001  285.879395  273.931396   
                    -100.25         0   0001  285.861816  273.804443   
                    -100.00         0   0001  285.912598  273.697021   
                    -99.75          0   0001  285.930176  273.673584   
                    -99.50          0   0001  286.074707  273.825928   

                                       sp       tcwv  cape  cin  
valid_time latitude longitude                                    
1950-04-28 36.5     -100.50    91507.5625  12.176069   0.0  NaN  
                    -100.25    92061.5625  12.703413   0.0  NaN  
                    -100.00    92680.5625  13.299116   0.0  NaN  
                    -99.75     93148.5625  13.744429   0.0  NaN  
                    -99.50     93723.5625  14.396772   0.0  NaN

In [5]:
from process_noaa_data import open_nc

ds = open_nc("file_1.nc")
print(ds)

<xarray.Dataset> Size: 149MB
Dimensions:     (valid_time: 264, latitude: 105, longitude: 168)
Coordinates:
    number      int64 8B ...
  * valid_time  (valid_time) datetime64[ns] 2kB 1950-08-02 ... 1950-08-31T23:...
  * latitude    (latitude) float64 840B 50.88 50.63 50.38 ... 25.38 25.13 24.88
  * longitude   (longitude) float64 1kB -112.4 -112.2 -111.9 ... -70.92 -70.67
    expver      (valid_time) <U4 4kB ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 19MB ...
    d2m         (valid_time, latitude, longitude) float32 19MB ...
    sp          (valid_time, latitude, longitude) float32 19MB ...
    tcwv        (valid_time, latitude, longitude) float32 19MB ...
    cape        (valid_time, latitude, longitude) float32 19MB ...
    cin         (valid_time, latitude, longitude) float32 19MB ...
    u10         (valid_time, latitude, longitude) float32 19MB ...
    v10         (valid_time, latitude, longitude) float32 19MB ...
Attributes:
    GRIB_centre:   

In [21]:
pt = ds.sel(valid_time=ds.valid_time[0], latitude=35.12, longitude=-99.20, method="nearest")
print(float(pt["cape"]), float(pt["cin"]), float(pt["t2m"]), float(pt["d2m"]), float(pt['sp']), float(pt['tcwv']))


104.625 267.62744140625 296.66259765625 292.56298828125 96057.375 43.51437759399414
